# Caso A — Abastecimiento · Modelado y evaluación

> **Objetivo.** Entrenar el forecast probabilístico de demanda semanal por SKU-tienda y la política de pedido, y evaluar su desempeño e impacto.

> **Entradas.** La misma tabla maestra semanal del EDA, cargada por el catálogo de Kedro (`data/01_raw`).

> **Salidas.** Métricas de forecast, comparación de modelos, figuras de desempeño, importancia de features y ahorro de la política newsvendor.

> **Cómo ejecutar.** `Restart & Run All`; es determinista (semillas fijas). Reutiliza `tostao_ml` —no reimplementa lógica—; este notebook corre en paralelo al pipeline `caso_a` y produce las mismas lecturas del reporte.

## 1. Datos: tabla maestra semanal

Mismo cruce que el EDA, agregado a grano semanal SKU-tienda.

In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from tostao_ml.cases import masters

PROJECT = Path.cwd().parents[1] if Path.cwd().name.startswith('caso') else Path.cwd()
bootstrap_project(PROJECT)
with KedroSession.create(project_path=PROJECT) as session:
    catalog = session.load_context().catalog
    master_d, _ = masters.build_master_a(
        catalog.load('a_ventas_historicas'), catalog.load('a_catalogo_productos'),
        catalog.load('a_maestro_tiendas'), catalog.load('a_inventario_actual'),
        catalog.load('a_ground_truth_trends'))
weekly = masters.aggregate_weekly_a(master_d)
print('Master semanal:', weekly.shape)
weekly.head()

## 2. Estrategia de modelado y validación

- **Partición:** holdout **temporal** (últimas 3 semanas como test), sin fuga de futuro.
- **Validación / HPO:** **walk-forward** (`TimeSeriesSplit`) con Optuna (TPE).
- **Modelo:** gradient boosting **cuantílico** (pérdida pinball), un estimador por cuantil (0.1/0.5/0.9) para obtener intervalos.
- **Comparación:** en el holdout se enfrentan Ridge, GBR, el cuantílico y un ensemble.

In [ ]:
from tostao_ml.cases.caso_a import run_case_a

result = run_case_a(weekly, tune=True, n_trials=20)

## 3. Métricas de forecast (holdout)

Lectura, no definición: WAPE es el error porcentual robusto; R² la varianza explicada; PICP la cobertura de los intervalos.

In [ ]:
import pandas as pd

pd.DataFrame([result.metrics]).T.rename(columns={0: 'valor'}).round(4)

## 4. Comparación de modelos

Más de un modelo validado: se elige por menor WAPE en el holdout.

In [ ]:
from tostao_ml.framework.evaluation import performance

if result.comparison is not None:
    display(result.comparison.round(4))
    performance.model_comparison_bar(
        result.comparison['wape'].to_dict(), 'WAPE (menor es mejor)').show()

## 5. Desempeño en el holdout

Predicho vs. real, residuales y su distribución: buscamos que los residuales no tengan patrón.

In [ ]:
test = result.test
performance.pred_vs_actual(test['unidades_vendidas'], test['pred']).show()
performance.residuals_vs_pred(test['unidades_vendidas'], test['pred']).show()
performance.residual_hist(test['unidades_vendidas'], test['pred']).show()

## 6. Importancia de features (permutación)

Qué variables sostienen el pronóstico: caída del RMSE al permutar cada feature.

In [ ]:
from tostao_ml.framework.evaluation import metrics
from tostao_ml.framework.interpret import permutation_bar, permutation_importance

imp = permutation_importance(
    result.model, test[result.feature_names], test['unidades_vendidas'],
    metrics.rmse, n_repeats=3)
permutation_bar(imp).show()

## 7. Impacto de negocio (newsvendor)

Del pronóstico a la decisión: la política óptima frente a pedir lo del período anterior.

In [ ]:
performance.model_comparison_bar(
    {'Política óptima': result.cost_model, 'Política ingenua': result.cost_naive},
    'costo esperado (menor es mejor)').show()
result.orders.head(15).round(2)

## 8. Lectura interpretada

La misma narración del reporte, derivada de las cifras reales del run.

In [ ]:
from IPython.display import Markdown
from tostao_ml.cases import storytelling as st

Markdown(st.interpret_model_a(result).to_markdown())

## Conclusión

El pronóstico supera al baseline de persistencia y su incertidumbre alimenta una política de pedido que reduce el costo esperado de faltante+sobrante. Los intervalos aún sub-cubren, por lo que se recomienda calibrarlos antes de fijar niveles de servicio.